# roomdiff: what changed in the room?
Compare two phone videos of the same room and find what was **moved, added or removed** in 3D.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom.
Filming tips: [docs/CAPTURE_GUIDE.md](https://github.com/amirhebeish/roomdiff/blob/main/docs/CAPTURE_GUIDE.md)

## 1 · Install (about 1 minute)

In [ ]:
REPO = "https://github.com/amirhebeish/roomdiff"
!git clone -q $REPO /content/roomdiff
%cd /content/roomdiff
!git clone -q --depth 1 https://github.com/facebookresearch/vggt third_party/vggt
!pip install -q -e . einops safetensors huggingface_hub
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: switch the runtime to a T4 GPU")

## 2 · Upload your two videos
Run the cell, then pick the **before** video first and the **after** video second when asked.

In [ ]:
from google.colab import files
import shutil
print("Choose the BEFORE video"); up = files.upload(); shutil.move(next(iter(up)), "/content/before.mp4")
print("Choose the AFTER video");  up = files.upload(); shutil.move(next(iter(up)), "/content/after.mp4")

*(Optional)* no videos yet? Run the synthetic benchmark instead. It needs no GPU and prints a ground-truth scorecard:

In [ ]:
# !python -m roomdiff demo --out /content/runs/synthetic

## 3 · Run (several minutes the first time: model downloads)

In [ ]:
OUT = "/content/runs/my_room"
!python -m roomdiff run --before /content/before.mp4 --after /content/after.mp4 --out $OUT --frames 24

## 4 · Results

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown("**Expected vs actual**: what video 1 predicts each camera of video 2 should see, vs what it saw"))
display(Image(f"{OUT}/figures/expected_vs_actual.gif"))
display(Markdown(open(f"{OUT}/report.md").read().replace("](figures/", f"]({OUT}/figures/")))

### Interactive 3D viewer

In [ ]:
import subprocess, time
from google.colab import output
subprocess.Popen(["python", "-m", "http.server", "8765", "--directory", OUT])
time.sleep(1)
output.serve_kernel_port_as_iframe(8765, path="/viewer.html", height=820)

## 5 · Download everything
`viewer.html` is a single self-contained file: open it in any browser.

In [ ]:
!cd /content/runs && zip -qr my_room_results.zip my_room -x "my_room/cache/*" "my_room/frames/*"
files.download("/content/runs/my_room_results.zip")

## 6 · *(Optional)* Re-run only the reasoning with different settings (seconds, no GPU)
The reconstruction and detections are cached in `runs/my_room/cache`.

In [ ]:
# looser depth tolerance (if you see false changes on walls) and a known phone height
!python -m roomdiff analyze --out $OUT --tau 0.08 --camera-height 1.3